[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sausterm/omniscience-research/blob/main/notebooks/portfolio_demo.ipynb)
[![View on GitHub](https://img.shields.io/badge/GitHub-View%20Source-blue?logo=github)](https://github.com/sausterm/omniscience-research/blob/main/notebooks/portfolio_demo.ipynb)

# Riemannian Portfolio Optimization
## Covariance Estimation on the SPD Manifold GL$^+$(d)/SO(d)

**OmniSciences LLC** | [omnisciences.io](https://omnisciences.io)

---

This notebook demonstrates the OmniSciences Riemannian Portfolio API, which performs covariance estimation, portfolio optimization, and regime detection on the manifold of symmetric positive-definite (SPD) matrices.

Standard (Euclidean) covariance estimators can produce ill-conditioned matrices, negative eigenvalues under perturbation, and unstable portfolio weights. Our Riemannian approach computes the **Frechet mean** on the SPD manifold, guaranteeing positive-definiteness and dramatically improving condition numbers.

> **API Key**: Reach out to sloan@omnisciences.org for a demo API key.

## 1. Setup

In [ ]:
import requests
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
%matplotlib inline

plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['font.size'] = 12

# ── Configuration ──────────────────────────────────────────────
API_URL = "https://api.omnisciences.io"
API_KEY = "your-api-key-here"  # Reach out to sloan@omnisciences.org for a demo key
# ──────────────────────────────────────────────────────────────

HEADERS = {"X-API-Key": API_KEY, "Content-Type": "application/json"}


def api_call(endpoint, payload):
    """POST to the Portfolio API and return the JSON response."""
    url = f"{API_URL}/portfolio/{endpoint}"
    resp = requests.post(url, json=payload, headers=HEADERS, timeout=60)
    resp.raise_for_status()
    return resp.json()


# Quick connectivity check
try:
    r = requests.get(f"{API_URL}/portfolio/health", headers=HEADERS, timeout=5)
    info = r.json()
    print(f"API status:   {info['status']}")
    print(f"Manifold:     {info['manifold']}")
    print(f"Methods:      {', '.join(info['supported_methods'])}")
    print(f"Strategies:   {', '.join(info['supported_strategies'])}")
except Exception as e:
    print(f"Cannot reach API at {API_URL}")
    print(f"Error: {e}")

## 2. Generate Sample Data

We create synthetic daily returns for 10 assets with realistic parameters: ~8% annualized return, ~20% annualized volatility, and moderate cross-correlations. This avoids any dependency on market data providers.

In [ ]:
np.random.seed(42)

n_assets = 10
n_days = 252  # 1 year of daily data

asset_names = [f"Asset_{i+1}" for i in range(n_assets)]

# Annualized parameters
annual_return = 0.08
annual_vol = 0.20
daily_mu = annual_return / 252
daily_vol = annual_vol / np.sqrt(252)

# Build a realistic correlation matrix with sector structure
# Assets 0-3: "Tech" sector (higher internal correlation)
# Assets 4-6: "Energy" sector
# Assets 7-9: "Healthcare" sector
base_corr = np.eye(n_assets) * 0.6 + 0.4 * np.ones((n_assets, n_assets))
for sector in [(0,4), (4,7), (7,10)]:
    s, e = sector
    base_corr[s:e, s:e] = np.eye(e-s) * 0.3 + 0.7 * np.ones((e-s, e-s))

# Convert correlation to covariance
vols = daily_vol * (0.8 + 0.4 * np.random.rand(n_assets))  # vary vol per asset
cov_true = np.outer(vols, vols) * base_corr

# Generate returns
returns = np.random.multivariate_normal(
    mean=np.full(n_assets, daily_mu),
    cov=cov_true,
    size=n_days
)

# Inject a volatility regime change at day 160
returns[160:200] *= 2.5  # high-vol episode

df = pd.DataFrame(returns, columns=asset_names)
print(f"Returns shape: {df.shape}")
print(f"\nAnnualized stats (first 5 assets):")
print(f"  Mean return: {(df.iloc[:, :5].mean() * 252).round(4).tolist()}")
print(f"  Volatility:  {(df.iloc[:, :5].std() * np.sqrt(252)).round(4).tolist()}")

# Plot cumulative returns
fig, ax = plt.subplots(figsize=(12, 5))
(1 + df).cumprod().plot(ax=ax, alpha=0.8)
ax.axvspan(160, 200, alpha=0.15, color='red', label='High-vol regime')
ax.set_xlabel('Trading Day')
ax.set_ylabel('Cumulative Return')
ax.set_title('Synthetic Portfolio: 10 Assets with Embedded Regime Change')
ax.legend(loc='upper left', fontsize=8, ncol=3)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 3. Covariance Estimation

We compare **Riemannian (Frechet mean)** vs **Euclidean (arithmetic mean)** covariance estimation. The Frechet mean is the point on the SPD manifold that minimizes the sum of squared geodesic distances -- it always produces a well-conditioned, positive-definite matrix.

In [ ]:
returns_list = returns.tolist()

# Riemannian covariance (Frechet mean on SPD manifold)
riem_resp = api_call("covariance", {
    "returns": returns_list,
    "method": "frechet",
    "window": 60,
    "ticker_names": asset_names
})

# Euclidean covariance (arithmetic mean)
euc_resp = api_call("covariance", {
    "returns": returns_list,
    "method": "arithmetic",
    "window": 60,
    "ticker_names": asset_names
})

cov_riem = np.array(riem_resp["covariance"])
cov_euc = np.array(euc_resp["covariance"])

print(f"{'Metric':<25} {'Riemannian':>15} {'Euclidean':>15}")
print("=" * 55)
print(f"{'Condition Number':<25} {riem_resp['condition_number']:>15.1f} {euc_resp['condition_number']:>15.1f}")
print(f"{'Max Eigenvalue':<25} {riem_resp['eigenvalues'][0]:>15.6f} {euc_resp['eigenvalues'][0]:>15.6f}")
print(f"{'Min Eigenvalue':<25} {riem_resp['eigenvalues'][-1]:>15.6f} {euc_resp['eigenvalues'][-1]:>15.6f}")
print(f"{'Eigenvalue Ratio':<25} {riem_resp['eigenvalues'][0]/riem_resp['eigenvalues'][-1]:>15.1f} {euc_resp['eigenvalues'][0]/euc_resp['eigenvalues'][-1]:>15.1f}")
print(f"\nCondition number improvement: {euc_resp['condition_number']/riem_resp['condition_number']:.1f}x")

In [ ]:
# Side-by-side covariance heatmaps
fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))

vmin = min(cov_riem.min(), cov_euc.min())
vmax = max(cov_riem.max(), cov_euc.max())

im0 = axes[0].imshow(cov_riem, cmap='RdBu_r', vmin=vmin, vmax=vmax, aspect='equal')
axes[0].set_title(f'Riemannian (Frechet Mean)\ncond = {riem_resp["condition_number"]:.1f}',
                  fontsize=13, fontweight='bold')
axes[0].set_xticks(range(n_assets))
axes[0].set_xticklabels(asset_names, rotation=45, ha='right', fontsize=8)
axes[0].set_yticks(range(n_assets))
axes[0].set_yticklabels(asset_names, fontsize=8)

im1 = axes[1].imshow(cov_euc, cmap='RdBu_r', vmin=vmin, vmax=vmax, aspect='equal')
axes[1].set_title(f'Euclidean (Arithmetic Mean)\ncond = {euc_resp["condition_number"]:.1f}',
                  fontsize=13, fontweight='bold')
axes[1].set_xticks(range(n_assets))
axes[1].set_xticklabels(asset_names, rotation=45, ha='right', fontsize=8)
axes[1].set_yticks(range(n_assets))
axes[1].set_yticklabels(asset_names, fontsize=8)

fig.colorbar(im1, ax=axes, shrink=0.8, label='Covariance')
plt.suptitle('Covariance Matrix Comparison', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

## 4. Portfolio Optimization

We optimize a **minimum-variance** portfolio using both Riemannian and Euclidean covariance estimates. The Riemannian approach typically produces more stable, less concentrated weight allocations.

In [ ]:
# Riemannian min-variance
opt_riem = api_call("optimize", {
    "returns": returns_list,
    "strategy": "min_variance",
    "method": "riemannian",
    "window": 60,
    "ticker_names": asset_names
})

# Euclidean min-variance
opt_euc = api_call("optimize", {
    "returns": returns_list,
    "strategy": "min_variance",
    "method": "euclidean",
    "window": 60,
    "ticker_names": asset_names
})

w_riem = np.array(opt_riem["weights"])
w_euc = np.array(opt_euc["weights"])

print(f"{'Metric':<30} {'Riemannian':>15} {'Euclidean':>15}")
print("=" * 60)
print(f"{'Expected Risk (daily vol)':<30} {opt_riem['expected_risk']:>15.6f} {opt_euc['expected_risk']:>15.6f}")
print(f"{'Condition Number':<30} {opt_riem['condition_number']:>15.1f} {opt_euc['condition_number']:>15.1f}")
print(f"{'Max |weight|':<30} {np.max(np.abs(w_riem)):>15.4f} {np.max(np.abs(w_euc)):>15.4f}")
print(f"{'Short exposure (sum of neg)':<30} {w_riem[w_riem<0].sum():>15.4f} {w_euc[w_euc<0].sum():>15.4f}")
print(f"{'HHI concentration':<30} {np.sum(w_riem**2):>15.4f} {np.sum(w_euc**2):>15.4f}")

In [ ]:
# Weight comparison bar chart
fig, ax = plt.subplots(figsize=(12, 5))

x = np.arange(n_assets)
width = 0.35

bars1 = ax.bar(x - width/2, w_riem, width, label='Riemannian', color='#2980b9',
               edgecolor='black', linewidth=0.5)
bars2 = ax.bar(x + width/2, w_euc, width, label='Euclidean', color='#e74c3c',
               edgecolor='black', linewidth=0.5)

ax.set_xlabel('Asset')
ax.set_ylabel('Weight')
ax.set_title('Minimum-Variance Portfolio Weights: Riemannian vs Euclidean',
             fontsize=14, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(asset_names, rotation=45, ha='right')
ax.legend(fontsize=12)
ax.axhline(y=0, color='black', linewidth=0.5)
ax.axhline(y=1/n_assets, color='gray', linestyle='--', alpha=0.5, label='Equal weight')
ax.grid(True, alpha=0.2, axis='y')
plt.tight_layout()
plt.show()

print("The Riemannian weights are typically more stable and less extreme.")
print("This translates to lower turnover and transaction costs in practice.")

## 5. Regime Detection

The API detects structural changes in the covariance matrix by measuring **geodesic distances** between consecutive rolling windows. Geodesic distance on the SPD manifold is more sensitive to eigenvalue changes than the Euclidean (Frobenius) norm, enabling earlier detection of regime shifts.

In [ ]:
regime_resp = api_call("regime-detection", {
    "returns": returns_list,
    "window": 60,
    "step": 5,
    "threshold": 2.0,
    "ticker_names": asset_names
})

geo_dist = np.array(regime_resp["geodesic_distances"])
euc_dist = np.array(regime_resp["euclidean_distances"])
geo_changes = regime_resp["geodesic_changes"]
euc_changes = regime_resp["euclidean_changes"]

print(f"Geodesic regime changes detected at indices: {geo_changes}")
print(f"Euclidean regime changes detected at indices: {euc_changes}")
print(f"Total geodesic detections: {regime_resp['n_detections']}")

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 7), sharex=True)

# Top: Distance time series
t = np.arange(len(geo_dist))
axes[0].plot(t, geo_dist, 'b-', lw=1.5, label='Geodesic distance', alpha=0.9)
axes[0].plot(t, euc_dist, 'r-', lw=1.5, label='Euclidean distance', alpha=0.7)

# Mark the true regime change region
# The regime change was injected at days 160-200; adjust for rolling window offset
axes[0].axvspan(0, len(geo_dist), alpha=0, label='_')  # placeholder

# Threshold lines
geo_thresh = np.mean(geo_dist) + 2.0 * np.std(geo_dist)
euc_thresh = np.mean(euc_dist) + 2.0 * np.std(euc_dist)
axes[0].axhline(y=geo_thresh, color='blue', linestyle='--', alpha=0.4, label='Geo threshold')
axes[0].axhline(y=euc_thresh, color='red', linestyle='--', alpha=0.4, label='Euc threshold')

axes[0].set_ylabel('Distance')
axes[0].set_title('Regime Detection: Geodesic vs Euclidean Distance', fontsize=14, fontweight='bold')
axes[0].legend(loc='upper right')
axes[0].grid(True, alpha=0.3)

# Bottom: Detected change points
axes[1].eventplot([geo_changes], lineoffsets=1.5, linelengths=0.8,
                  colors='blue', label='Geodesic detections')
axes[1].eventplot([euc_changes], lineoffsets=0.5, linelengths=0.8,
                  colors='red', label='Euclidean detections')
axes[1].set_yticks([0.5, 1.5])
axes[1].set_yticklabels(['Euclidean', 'Geodesic'])
axes[1].set_xlabel('Window Index')
axes[1].set_title('Detected Regime Change Points', fontsize=13)
axes[1].legend(loc='upper right')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\nGeodesic distance is more sensitive to eigenvalue shifts in the covariance")
print("structure, often detecting regime changes earlier than Euclidean norms.")

## 6. Full Method Comparison

The `/portfolio/compare` endpoint benchmarks all three covariance methods (Euclidean, Ledoit-Wolf, Riemannian) head-to-head on the same data.

In [ ]:
compare_resp = api_call("compare", {
    "returns": returns_list,
    "window": 60,
    "ticker_names": asset_names
})

methods = compare_resp["methods"]

# Build comparison table
rows = []
for method_name in ["euclidean", "ledoit_wolf", "riemannian"]:
    m = methods[method_name]
    rows.append({
        "Method": method_name.replace("_", " ").title(),
        "Condition Number": f"{m['condition_number']:.1f}",
        "Min Eigenvalue": f"{m['min_eigenvalue']:.2e}",
        "Eigenvalue Ratio": f"{m['eigenvalue_ratio']:.1f}",
        "Min-Var Risk": f"{m['min_variance_risk']:.6f}",
        "Max |Weight|": f"{m['max_abs_weight']:.4f}",
    })

comparison_df = pd.DataFrame(rows).set_index("Method")
print(comparison_df.to_string())

In [ ]:
# Visualize key metrics
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

method_labels = ["Euclidean", "Ledoit-Wolf", "Riemannian"]
method_keys = ["euclidean", "ledoit_wolf", "riemannian"]
colors = ["#e74c3c", "#f39c12", "#2980b9"]

# Condition Number
cond_vals = [methods[k]["condition_number"] for k in method_keys]
axes[0].bar(method_labels, cond_vals, color=colors, edgecolor='black', linewidth=0.5)
axes[0].set_ylabel('Condition Number')
axes[0].set_title('Matrix Conditioning\n(lower = more stable)', fontweight='bold')
axes[0].grid(True, alpha=0.3, axis='y')

# Min-Variance Risk
risk_vals = [methods[k]["min_variance_risk"] for k in method_keys]
axes[1].bar(method_labels, risk_vals, color=colors, edgecolor='black', linewidth=0.5)
axes[1].set_ylabel('Portfolio Risk (daily vol)')
axes[1].set_title('Minimum-Variance Risk\n(lower = better)', fontweight='bold')
axes[1].grid(True, alpha=0.3, axis='y')

# Max Weight Concentration
weight_vals = [methods[k]["max_abs_weight"] for k in method_keys]
axes[2].bar(method_labels, weight_vals, color=colors, edgecolor='black', linewidth=0.5)
axes[2].set_ylabel('Max |Weight|')
axes[2].set_title('Weight Concentration\n(lower = more diversified)', fontweight='bold')
axes[2].grid(True, alpha=0.3, axis='y')

plt.suptitle('Method Comparison: Euclidean vs Ledoit-Wolf vs Riemannian',
             fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

## 7. Key Takeaways

| Feature | Euclidean | Ledoit-Wolf | Riemannian (Ours) |
|---------|:---------:|:-----------:|:-----------------:|
| **SPD guarantee** | No | Partial | **Always** |
| **Condition number** | High | Medium | **Low** |
| **Weight stability** | Unstable | Better | **Best** |
| **Regime detection** | Frobenius only | N/A | **Geodesic distance** |
| **Theoretical foundation** | Flat geometry | Shrinkage heuristic | **Riemannian manifold** |

### Why Riemannian?

- **Lower condition number**: The Frechet mean on GL$^+$(d)/SO(d) naturally regularizes the covariance estimate without ad-hoc shrinkage, producing better-conditioned matrices.
- **Guaranteed positive-definiteness**: Every point on the SPD manifold is positive-definite by construction. No eigenvalue clipping or post-hoc fixes needed.
- **Lower turnover**: More stable covariance estimates lead to smoother weight trajectories, reducing transaction costs.
- **Earlier regime detection**: Geodesic distance is more sensitive to eigenvalue changes than the Frobenius norm, catching structural shifts sooner.

### API Access

- **Free tier**: 100 requests/month
- **Pro tier**: Unlimited requests, batch processing, priority support
- **Enterprise**: On-premise deployment, custom manifolds

### Contact

**OmniSciences LLC**  
Email: sloan@omnisciences.org  
Web: [omnisciences.io](https://omnisciences.io)  

*Patent pending. (c) 2026 OmniSciences LLC.*